In [0]:
# ============================================
# NOTEBOOK 1: Bronze Layer - Raw Data Ingestion
# PROJECT: Wikipedia Clickstream Analytics
# ============================================

# Step 1: Verify files are uploaded correctly
print("Checking uploaded files...")
display(dbutils.fs.ls("/Volumes/workspace/default/wikidata/"))

Checking uploaded files...


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/wikidata/clickstream-enwiki-2024-10.tsv.gz,clickstream-enwiki-2024-10.tsv.gz,500995036,1777008634000
dbfs:/Volumes/workspace/default/wikidata/pageviews-20241101-010000.gz,pageviews-20241101-010000.gz,42432354,1777008570000


In [0]:
# ============================================
# BRONZE: Load Raw Clickstream Data
# ============================================
from pyspark.sql.functions import current_timestamp

clickstream_path = "/Volumes/workspace/default/wikidata/clickstream-enwiki-2024-10.tsv.gz"

bronze_clickstream = spark.read.option("sep", "\t").option("header", "false").option("inferSchema", "true").csv(clickstream_path)

bronze_clickstream = bronze_clickstream.withColumnRenamed("_c0", "source_page").withColumnRenamed("_c1", "target_page").withColumnRenamed("_c2", "click_type").withColumnRenamed("_c3", "click_count")

bronze_clickstream = bronze_clickstream.withColumn("ingested_at", current_timestamp())

print("Clickstream record count:", bronze_clickstream.count())
display(bronze_clickstream.limit(5))

Clickstream record count: 34170973


source_page,target_page,click_type,click_count,ingested_at
other-empty,Main_Page,external,120677075,2026-04-29T02:34:46.303Z
Chinese_water_dragon,Arthropod,link,12,2026-04-29T02:34:46.303Z
other-search,Lyle_and_Erik_Menendez,external,7756271,2026-04-29T02:34:46.303Z
Chinese_water_snake,Snake,link,12,2026-04-29T02:34:46.303Z
other-internal,Main_Page,external,7469673,2026-04-29T02:34:46.303Z


In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Read each file separately with correct exact names
files = [
    ("/Volumes/workspace/default/wikidata/pageviews-20241101-000000.gz", 0),
    ("/Volumes/workspace/default/wikidata/pageviews-20241101-010000.gz", 1),
    ("/Volumes/workspace/default/wikidata/pageviews-20241101-020000.gz", 2),
    ("/Volumes/workspace/default/wikidata/pageviews-20241101-040000.gz", 4),
    ("/Volumes/workspace/default/wikidata/pageviews-20241101-060000.gz", 6),
    ("/Volumes/workspace/default/wikidata/pageviews-20241101-080000.gz", 8),
]

dfs = []
for path, hour in files:
    df = spark.read.option("sep", " ").option("header", "false").option("inferSchema", "true").csv(path)
    df = df.withColumnRenamed("_c0", "language_code")
    df = df.withColumnRenamed("_c1", "article_title")
    df = df.withColumnRenamed("_c2", "view_count")
    df = df.withColumnRenamed("_c3", "response_size")
    df = df.withColumn("hour_of_day", lit(hour))
    df = df.withColumn("ingested_at", current_timestamp())
    dfs.append(df)
    print(f"✅ Hour {hour} loaded!")

bronze_pageviews = dfs[0]
for df in dfs[1:]:
    bronze_pageviews = bronze_pageviews.union(df)

print("Total Pageview records:", bronze_pageviews.count())
display(bronze_pageviews.select("article_title", "view_count", "hour_of_day").limit(10))

✅ Hour 0 loaded!
✅ Hour 1 loaded!
✅ Hour 2 loaded!
✅ Hour 4 loaded!
✅ Hour 6 loaded!
✅ Hour 8 loaded!
Total Pageview records: 30754778


article_title,view_count,hour_of_day
Category:Drafts,1,0
File:Crystal_Clear_action_run_approved.svg,1,0
File:George_Town_Conurbation_map.svg,1,0
File:Hagelslag_chocolate_sprinkles.jpg,1,0
File:Thailand_Pattani_location_map.svg,1,0
"File:Venus_symbol_(bold,_white).svg",1,0
File:Wikifunctions-logo.svg,2,0
Help:Contents,1,0
Special:MyLanguage/Help:Multilingual,1,0
Special:MyLanguage/Wikifunctions:About,1,0


In [0]:
# ============================================
# BRONZE: Save to Delta Lake Tables
# ============================================
from pyspark.sql.functions import current_timestamp

# Reload clickstream
clickstream_path = "/Volumes/workspace/default/wikidata/clickstream-enwiki-2024-10.tsv.gz"
bronze_clickstream = spark.read.option("sep", "\t").option("header", "false").option("inferSchema", "true").csv(clickstream_path)
bronze_clickstream = bronze_clickstream.withColumnRenamed("_c0", "source_page").withColumnRenamed("_c1", "target_page").withColumnRenamed("_c2", "click_type").withColumnRenamed("_c3", "click_count").withColumn("ingested_at", current_timestamp())

print("Clickstream records:", bronze_clickstream.count())

# Save clickstream
bronze_clickstream.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_clickstream")
print("✅ bronze_clickstream table saved!")

# Save pageviews with hour_of_day column
bronze_pageviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_pageviews")
print("✅ bronze_pageviews table saved!")
print("🎉 Bronze Layer Complete!")

Clickstream records: 34170973
✅ bronze_clickstream table saved!
✅ bronze_pageviews table saved!
🎉 Bronze Layer Complete!


In [0]:
files = dbutils.fs.ls("/Volumes/workspace/default/wikidata/")
for f in files:
    print(f.name)

clickstream-enwiki-2024-10.tsv.gz
pageviews-20241101-000000.gz
pageviews-20241101-010000.gz
pageviews-20241101-020000.gz
pageviews-20241101-040000.gz
pageviews-20241101-060000.gz
pageviews-20241101-080000.gz
